In [3]:
import pandas as pd
import numpy as np

# --- 1.0 Load the Data ---
# Loading from Sheet 2 of your specific file
file_path = r"C:\Users\HP\OneDrive - PESUNIVERSITY\Desktop\PDA - RP DETAILS FINAL DATA.xlsx"
df = pd.read_excel(file_path, sheet_name='Sheet2')

# Filter for your specific study period (2016-2025)
df = df[(df['Year End'] >= 2016) & (df['Year End'] <= 2025)].copy()

# --- 1.1 Set the Panel Structure ---
# Setting Company (cap_code equivalent) and Year End as the MultiIndex
df = df.set_index(['Company Name', 'Year End'])

# Verification of Panel Balance
# Since you have 78 companies and 10 years, the count should be exactly 780
expected_obs = 78 * 10
actual_obs = len(df)
is_balanced = df.index.is_unique and actual_obs == expected_obs

print(f"Panel Structure Check:")
print(f"Total Observations: {actual_obs}")
print(f"Is the panel balanced? {is_balanced}")

# --- 1.2 Declare Your Variable Lists ---
# Organizing variables into functional groups for the regression
# Dependent variable: ROA
dep_var = ['ROA']

# Main independent variables: LEV (Leverage), LEV SQUARED (Quadratic term)
main_indep_vars = ['LEV', 'LEV SQUARE']

# Control variables: FIRM SIZE, TANGIBILITY, SALES GROWTH, LIQUIDITY
control_vars = ['FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']

all_model_vars = dep_var + main_indep_vars + control_vars

# --- 1.3 Final Data Checks ---

# A. Variance Check: Ensure no column is constant (zero variance)
# A variable with zero variance cannot be used in a regression
variances = df[all_model_vars].var()
print("\nVariance Check (Should be > 0):")
print(variances)

# B. Leverage Bounds Check: LEV should be between 0 and 1
lev_min = df['LEV'].min()
lev_max = df['LEV'].max()
print(f"\nLEV Range Check: Min={lev_min:.4f}, Max={lev_max:.4f}")
if lev_min < 0 or lev_max > 1:
    print("WARNING: LEV values found outside the 0-1 ratio range.")

# C. ROA Bounds Check: ROA should generally fall between -1 and +1
roa_min = df['ROA'].min()
roa_max = df['ROA'].max()
print(f"ROA Range Check: Min={roa_min:.4f}, Max={roa_max:.4f}")

# Review the prepared dataframe head
print("\nPanel Structure Prepared (First 5 rows):")
print(df[all_model_vars].head())

Panel Structure Check:
Total Observations: 780
Is the panel balanced? True

Variance Check (Should be > 0):
ROA             0.007263
LEV             0.031610
LEV SQUARE      0.012971
FIRM SIZE       0.349254
TANGIBILITY     0.054533
SALES GROWTH    0.040729
LIQUIDITY       0.768726
dtype: float64

LEV Range Check: Min=0.0000, Max=1.1180
ROA Range Check: Min=-0.3887, Max=0.6418

Panel Structure Prepared (First 5 rows):
                            ROA       LEV  LEV SQUARE  FIRM SIZE  TANGIBILITY  \
Company Name Year End                                                           
A B B        2016      0.094977  0.152189    0.023161   3.595797     0.335476   
             2017      0.098539  0.142746    0.020376   3.629591     0.313282   
             2018      0.126145  0.001709    0.000003   3.607457     0.241036   
             2019      0.084697  0.003780    0.000014   3.554147     0.223654   
             2020      0.059735  0.015870    0.000252   3.564653     0.233683   

          

In [6]:
!pip install linearmodels

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.8 MB/s  0:00:00

   ---------------------------------------- 0/8 [wrapt]
   ---------------------------------------- 0/8 [wrapt]
   ---------------------------------------- 0/8 [wrapt]
   ----- ---------------------------------- 1/8 [typing-extensions]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 2/8 [narwhals]
   ---------- ----------------------------- 


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m pip install linearmodels

  Using cached pyhdfe-0.2.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached interface_meta-2.0.1-py3-none-any.whl.metadata (6.4 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 3.6 MB/s  0:00:00
Using cached formulaic-1.2.1-py3-none-any.whl (117 kB)
Using cached interface_meta-2.0.1-py3-none-any.whl (15 kB)
Using cached pyhdfe-0.2.0-py3-none-any.whl (19 kB)

   ---------- ----------------------------- 1/4 [pyhdfe]
   -------------------- ------------------- 2/4 [formulaic]
   -------------------- ------------------- 2/4 [formulaic]
   -------------------- ------------------- 2/4 [formulaic]
   -------------------- -----------------

In [5]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# --- Block 1.4: Winsorization ---

# Capping at the 1st and 99th percentiles (0.01 on each end)
# This handles the LEV values that are currently above 1.0 
# and the negative extremes in ROA.

# Winsorize ROA
df['ROA_w'] = winsorize(df['ROA'], limits=[0.01, 0.01])

# Winsorize LEV
df['LEV_w'] = winsorize(df['LEV'], limits=[0.01, 0.01])

# Recalculate the squared term based on the cleaned LEV
df['LEV_SQUARE_w'] = df['LEV_w'] ** 2

# --- Final Check of Cleaned Data ---

print("Winsorization Complete.")
print(f"Original LEV Max: {df['LEV'].max():.4f} | New LEV Max: {df['LEV_w'].max():.4f}")
print(f"Original ROA Min: {df['ROA'].min():.4f} | New ROA Min: {df['ROA_w'].min():.4f}")

# Display first 5 rows to confirm new columns
print("\nNew Columns Created:")
print(df[['ROA_w', 'LEV_w', 'LEV_SQUARE_w']].head())

Winsorization Complete.
Original LEV Max: 1.1180 | New LEV Max: 0.6848
Original ROA Min: -0.3887 | New ROA Min: -0.0159

New Columns Created:
                          ROA_w     LEV_w  LEV_SQUARE_w
Company Name Year End                                  
A B B        2016      0.094977  0.152189      0.023161
             2017      0.098539  0.142746      0.020376
             2018      0.126145  0.001709      0.000003
             2019      0.084697  0.003780      0.000014
             2020      0.059735  0.015870      0.000252


In [6]:
!pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# --- 2.1 Summary Statistics Table ---
# We use the winsorized variables (ROA_w, LEV_w) and original controls
summary_vars = ['ROA_w', 'LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
summary_stats = df[summary_vars].describe().T

# Adding Median (50%) to the table
summary_stats = summary_stats[['count', 'mean', '50%', 'std', 'min', 'max']]
summary_stats.columns = ['N', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']

print("--- Table 1: Summary Statistics ---")
print(summary_stats)

# --- 2.2 Correlation Matrix ---
# Exclude the dependent variable (ROA_w) to focus on multi-collinearity
corr_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
corr_matrix = df[corr_vars].corr(method='pearson')

print("\n--- Table 2: Correlation Matrix ---")
print(corr_matrix.round(3))

# --- 2.3 Variance Inflation Factor (VIF) ---
# Function to calculate VIF
def calculate_vif(dataframe, variables):
    X = dataframe[variables].dropna()
    X = sm.add_constant(X) # VIF requires an intercept
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data[vif_data['Variable'] != 'const'] # Return only variables

# Pass 1: All variables except the quadratic term (LEV_SQUARE_w)
vif_pass1_vars = ['LEV_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
vif_pass1 = calculate_vif(df, vif_pass1_vars)

# Pass 2: Including the quadratic term to see the inflation
vif_pass2_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
vif_pass2 = calculate_vif(df, vif_pass2_vars)

print("\n--- VIF Pass 1 (Excluding LEV² ---")
print(vif_pass1)

print("\n--- VIF Pass 2 (Including LEV² ---")
print(vif_pass2)

--- Table 1: Summary Statistics ---
                  N      Mean    Median   Std Dev       Min       Max
ROA_w         780.0  0.123130  0.119372  0.072684 -0.015941  0.358595
LEV_w         780.0  0.155604  0.084666  0.169737  0.000000  0.684839
LEV_SQUARE_w  780.0  0.052986  0.007168  0.088599  0.000000  0.469004
FIRM SIZE     780.0  3.462688  3.470568  0.590977  1.701741  4.978264
TANGIBILITY   780.0  0.424712  0.394586  0.233524  0.017886  1.324411
SALES GROWTH  778.0  0.093268  0.092348  0.201815 -1.000000  1.364998
LIQUIDITY     780.0  1.704308  1.450000  0.876770  0.030000  7.570000

--- Table 2: Correlation Matrix ---
              LEV_w  LEV_SQUARE_w  FIRM SIZE  TANGIBILITY  SALES GROWTH  \
LEV_w         1.000         0.943     -0.163        0.282        -0.075   
LEV_SQUARE_w  0.943         1.000     -0.192        0.215        -0.120   
FIRM SIZE    -0.163        -0.192      1.000       -0.293         0.027   
TANGIBILITY   0.282         0.215     -0.293        1.000         0

In [8]:
import statsmodels.api as sm

# --- 3.1 Define the Regression Variables ---
# Dependent variable: Winsorized ROA
y_pooled = df['ROA_w']

# Independent variables: Including the constant and all control variables
# Note: Python drops the 2 rows with missing Sales Growth automatically
independent_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
X_pooled = sm.add_constant(df[independent_vars])

# --- 3.2 Run the Pooled OLS Model ---
# This is a "naive" regression treating all observations as independent
model_pooled = sm.OLS(y_pooled, X_pooled, missing='drop')
results_pooled = model_pooled.fit()

# --- 3.3 Output the Results ---
print("--- TABLE 3: POOLED OLS (BASELINE MODEL) ---")
print(results_pooled.summary())

# Quick check for overall significance
print(f"\nOverall Model Significance (F-statistic): {results_pooled.fvalue:.4f}")
print(f"P-value of F-statistic: {results_pooled.f_pvalue:.4f}")
print(f"Adjusted R-squared: {results_pooled.rsquared_adj:.4f}")

--- TABLE 3: POOLED OLS (BASELINE MODEL) ---
                            OLS Regression Results                            
Dep. Variable:                  ROA_w   R-squared:                       0.285
Model:                            OLS   Adj. R-squared:                  0.279
Method:                 Least Squares   F-statistic:                     51.16
Date:                Sun, 03 May 2026   Prob (F-statistic):           4.78e-53
Time:                        20:04:59   Log-Likelihood:                 1069.0
No. Observations:                 778   AIC:                            -2124.
Df Residuals:                     771   BIC:                            -2091.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
con

In [9]:
import numpy as np
import pandas as pd
from linearmodels import PanelOLS, RandomEffects
import scipy.stats as stats
import statsmodels.api as sm

# --- 4.1 Run Both Models ---

# Setup Dependent and Independent Variables
# Note: PanelOLS and RandomEffects require the constant to be added manually
dependent = df['ROA_w']
exog_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
exog = sm.add_constant(df[exog_vars])

# 1. Fixed Effects (FE) Model
model_fe = PanelOLS(dependent, exog, entity_effects=True)
res_fe = model_fe.fit()

# 2. Random Effects (RE) Model
model_re = RandomEffects(dependent, exog)
res_re = model_re.fit()

# --- 4.2 The Hausman Test Function ---

def hausman(fe, re):
    """
    Computes the Hausman test to choose between FE and RE.
    Null Hypothesis: RE is consistent and efficient.
    """
    # Extract coefficients (excluding constant if necessary, though linearmodels handles this)
    b = fe.params
    B = re.params
    
    # Extract covariance matrices
    v_b = fe.cov
    v_B = re.cov
    
    # Calculate the test statistic (chi-square)
    # We use common coefficients found in both models
    common_cols = [c for c in b.index if c in B.index]
    diff = b[common_cols] - B[common_cols]
    v_diff = v_b.loc[common_cols, common_cols] - v_B.loc[common_cols, common_cols]
    
    # Chi-square statistic
    chi2 = np.dot(np.dot(diff.T, np.linalg.inv(v_diff)), diff)
    df_chi2 = len(common_cols)
    p_value = 1 - stats.chi2.cdf(chi2, df_chi2)
    
    return chi2, df_chi2, p_value

# Execute the test
chi2, df_hausman, p_val = hausman(res_fe, res_re)

# --- 4.3 Report the Results ---
print("--- HAUSMAN TEST RESULTS ---")
print(f"Chi-Squared Statistic: {chi2:.4f}")
print(f"Degrees of Freedom: {df_hausman}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("\nCONCLUSION: P < 0.05. Reject Null Hypothesis. Use FIXED EFFECTS.")
else:
    print("\nCONCLUSION: P > 0.05. Fail to Reject Null. RANDOM EFFECTS is acceptable.")

C:\Users\HP\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
C:\Users\HP\anaconda3\Lib\site-packages\linearmodels\panel\model.py:2751: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


--- HAUSMAN TEST RESULTS ---
Chi-Squared Statistic: 53.8635
Degrees of Freedom: 7
P-value: 0.0000

CONCLUSION: P < 0.05. Reject Null Hypothesis. Use FIXED EFFECTS.


In [10]:
# Find the specific rows that have missing values
missing_rows = df[df['SALES GROWTH'].isna()]

print("--- ROWS WITH MISSING SALES GROWTH ---")
if not missing_rows.empty:
    print(missing_rows[['SALES GROWTH']])
else:
    print("No NaN values found. They might be 'hidden' spaces.")
    # Check for empty strings or spaces
    missing_spaces = df[df['SALES GROWTH'].apply(lambda x: str(x).strip() == "")]
    print(missing_spaces[['SALES GROWTH']])

--- ROWS WITH MISSING SALES GROWTH ---
                           SALES GROWTH
Company Name     Year End              
Sanco Industries 2023               NaN
                 2025               NaN


In [11]:
import pandas as pd
import numpy as np
from linearmodels import PanelOLS
import statsmodels.api as sm

# --- 5.1 Set up the Two-Way Fixed Effects (TWFE) Model ---

# Dependent variable
y = df['ROA_w']

# Independent variables including the constant
# Note: linearmodels automatically handles the missing 'SALES GROWTH' values
exog_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
X = sm.add_constant(df[exog_vars])

# Define the Model: 
# entity_effects=True (Firm Fixed Effects)
# time_effects=True (Year Fixed Effects)
model_twfe = PanelOLS(y, X, entity_effects=True, time_effects=True)

# --- 5.2 Clustered Standard Errors ---
# We cluster at the firm (entity) level to solve for serial correlation and heteroskedasticity
results_twfe = model_twfe.fit(cov_type='clustered', cluster_entity=True)

# --- 5.3 Output the Results ---
print("--- TABLE 5: FINAL TWFE REGRESSION RESULTS ---")
print(results_twfe.summary)

# --- 5.4 Calculation of Optimal Leverage (Inverted-U Check) ---
beta1 = results_twfe.params['LEV_w']
beta2 = results_twfe.params['LEV_SQUARE_w']

if beta1 > 0 and beta2 < 0:
    optimal_lev = -beta1 / (2 * beta2)
    print(f"\n--- OPTIMAL CAPITAL STRUCTURE ANALYSIS ---")
    print(f"Optimal Leverage Ratio: {optimal_lev:.4f}")
    print(f"Sample Mean Leverage: {df['LEV_w'].mean():.4f}")
    
    if df['LEV_w'].mean() > optimal_lev:
        print("Finding: On average, firms in the sample are over-leveraged.")
    else:
        print("Finding: On average, firms in the sample have room to increase debt.")
else:
    print("\nNote: Relationship is not an Inverted-U. Optimal point calculation not applicable.")

C:\Users\HP\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


--- TABLE 5: FINAL TWFE REGRESSION RESULTS ---
                          PanelOLS Estimation Summary                           
Dep. Variable:                  ROA_w   R-squared:                        0.2879
Estimator:                   PanelOLS   R-squared (Between):             -0.5523
No. Observations:                 778   R-squared (Within):               0.2481
Date:                Sun, May 03 2026   R-squared (Overall):             -0.1861
Time:                        20:14:05   Log-likelihood                    1384.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      46.162
Entities:                          78   P-value                           0.0000
Avg Obs:                       9.9744   Distribution:                   F(6,685)
Min Obs:                       8.0000                                           
Max Obs:                      10.0000   F-statistic (robust): 

In [12]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

# --- 6.1 Test for Heteroskedasticity (Breusch-Pagan) ---
# We use the residuals from your final TWFE model
residuals = results_twfe.resids
exog_with_constant = sm.add_constant(df[['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']].dropna())

bp_test = het_breuschpagan(residuals, exog_with_constant)
print("--- 6.1 Heteroskedasticity Test (Breusch-Pagan) ---")
print(f"Lagrange Multiplier Statistic: {bp_test[0]:.4f}")
print(f"P-value: {bp_test[1]:.4f}")
# Interpretation: If P < 0.05, heteroskedasticity exists. (Handled by clustering).

# --- 6.2 Test for Serial Correlation (Wooldridge Test) ---
# For Python, we can check the Durbin-Watson statistic specifically for the panel residuals
from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(residuals)
print("\n--- 6.2 Serial Correlation (Durbin-Watson) ---")
print(f"Durbin-Watson Statistic: {dw_stat:.4f}")
# Interpretation: Values significantly below 2.0 indicate serial correlation. (Handled by clustering).

# --- 6.3 Test for Cross-Sectional Dependence ---
# This checks if firms are affected by common macro shocks
# In linearmodels, we can check the correlation of residuals across entities
def check_csd(resids):
    # Pivot residuals to have years as rows and entities as columns
    pivoted = resids.unstack(level=0)
    corr_matrix = pivoted.corr()
    avg_corr = (corr_matrix.sum().sum() - len(corr_matrix)) / (len(corr_matrix)**2 - len(corr_matrix))
    return avg_corr

avg_resid_corr = check_csd(residuals)
print("\n--- 6.3 Cross-Sectional Dependence (Residual Correlation) ---")
print(f"Average Correlation of Residuals: {avg_resid_corr:.4f}")
# Interpretation: High correlation suggests dependence. (Handled by Year Fixed Effects).

--- 6.1 Heteroskedasticity Test (Breusch-Pagan) ---
Lagrange Multiplier Statistic: 15.8385
P-value: 0.0146

--- 6.2 Serial Correlation (Durbin-Watson) ---
Durbin-Watson Statistic: 1.4803

--- 6.3 Cross-Sectional Dependence (Residual Correlation) ---
Average Correlation of Residuals: -0.0114


In [14]:
import pandas as pd
from linearmodels import PanelOLS
import statsmodels.api as sm

# Define the common regression function to ensure consistency across checks
def run_robustness_fe(data, label):
    y = data['ROA_w']
    exog_vars = ['LEV_w', 'LEV_SQUARE_w', 'FIRM SIZE', 'TANGIBILITY', 'SALES GROWTH', 'LIQUIDITY']
    X = sm.add_constant(data[exog_vars])
    
    # Using Two-Way Fixed Effects (Entity and Time) with Clustered Standard Errors
    model = PanelOLS(y, X, entity_effects=True, time_effects=True)
    results = model.fit(cov_type='clustered', cluster_entity=True)
    
    print(f"\n--- Robustness Check: {label} ---")
    print(results.summary.tables[1])
    return results

# --- 7.1 Sub-Period Analysis (Pre- vs. Post-COVID) ---
# Assuming your 'Year' column is the index level 1 or a column in the DF
# Pre-COVID: 2015-2019 | Post-COVID: 2020-2024
# Note: Since your data is 2016-2025, we will adapt to your actual range
pre_covid = df.xs(slice(2016, 2019), level='Year End', drop_level=False)
post_covid = df.xs(slice(2020, 2025), level='Year End', drop_level=False)

res_pre = run_robustness_fe(pre_covid, "Pre-COVID (2016-2019)")
res_post = run_robustness_fe(post_covid, "Post-COVID (2020-2025)")

# --- 7.2 Outlier Sensitivity (Excluding Top/Bottom 5% by Size) ---
lower_q = df['FIRM SIZE'].quantile(0.05)
upper_q = df['FIRM SIZE'].quantile(0.95)

filtered_df = df[(df['FIRM SIZE'] > lower_q) & (df['FIRM SIZE'] < upper_q)]
res_outlier = run_robustness_fe(filtered_df, "Excluding Size Outliers (Middle 90%)")


--- Robustness Check: Pre-COVID (2016-2019) ---
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            0.6413     0.1928     3.3257     0.0010      0.2613      1.0213
LEV_w            0.0279     0.0923     0.3026     0.7625     -0.1539      0.2097
LEV_SQUARE_w    -0.2692     0.1350    -1.9942     0.0473     -0.5352     -0.0032
FIRM SIZE       -0.1404     0.0564    -2.4878     0.0136     -0.2516     -0.0292
TANGIBILITY     -0.1117     0.0585    -1.9082     0.0576     -0.2270      0.0036
SALES GROWTH     0.0532     0.0266     1.9990     0.0468      0.0008      0.1057
LIQUIDITY        0.0044     0.0192     0.2309     0.8176     -0.0333      0.0422

--- Robustness Check: Post-COVID (2020-2025) ---


C:\Users\HP\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            0.5114     0.1726     2.9629     0.0032      0.1720      0.8508
LEV_w           -0.2084     0.0972    -2.1439     0.0327     -0.3995     -0.0173
LEV_SQUARE_w    -0.0055     0.1488    -0.0367     0.9707     -0.2980      0.2871
FIRM SIZE       -0.0799     0.0490    -1.6294     0.1041     -0.1763      0.0165
TANGIBILITY     -0.1795     0.0435    -4.1218     0.0000     -0.2651     -0.0939
SALES GROWTH     0.0967     0.0145     6.6713     0.0000      0.0682      0.1252
LIQUIDITY       -0.0053     0.0073    -0.7176     0.4734     -0.0197      0.0092

--- Robustness Check: Excluding Size Outliers (Middle 90%) ---
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat